# Notebook 07 — Noise Stability Score (NSS)

**QNLEP metric:** NSS measures how gracefully a trained circuit degrades under
depolarizing noise — a proxy for hardware deployability.

**Protocol:**
1. Train each circuit under **zero noise**.
2. Evaluate (frozen weights) with depolarizing noise injected after each gate
   at rates `p ∈ {0, 0.001, 0.005, 0.01, 0.05, 0.1}`.
3. Record raw `R²(p)` at each noise level.
4. Convert each `R²(p)` to a bounded skill score `clip((R² + 1) / 2, 0, 1)`.
5. Compute area under the bounded-skill curve (trapezoid rule).

```
NSS = trapz( shifted_skill(R²(p)), p_values ) / (p_max - p_min)
```

Raw `R²(p)` curves are still exported to `results/r2_curves.json`; the bounded
skill transform is only for the scalar score used in tables / NLPS.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import json, pathlib, os

from src.qnlep_registry import QNLEP_BUILDERS, param_shape
from src.qnlep_run_paths import protocol_results_dir, protocol_n_qubits, protocol_n_layers
from src.circuits.noisy_qnlep import bound_noisy_factories
from src.metrics.protocol_scores import noise_stability_score, shifted_r2_score

RUN_SEED = int(os.environ.get("QNLEP_SEED", "42"))
np.random.seed(RUN_SEED)
print("QNLEP_SEED =", RUN_SEED)


## 1. Configuration

In [ ]:
NOISE_LEVELS = [0.0, 0.001, 0.005, 0.01, 0.05, 0.1]
N_QUBITS     = protocol_n_qubits(3)
N_LAYERS     = protocol_n_layers(3)
N_TRAIN      = 80   # larger grid -> stabler clean R² before noise sweep (invalidate NSS cache if changed)
N_EVAL       = 150
N_STEPS      = 120
LR           = 0.1

FIGURES_DIR = pathlib.Path('../figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = protocol_results_dir()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
NSS_TRAIN_CACHE_PATH = RESULTS_DIR / 'nss_clean_train_cache.json'
NSS_USE_CLEAN_TRAIN_CACHE = True   # load `nss_clean_train_cache.json` when meta matches
NSS_FORCE_CLEAN_RETRAIN = False    # True: ignore cache for one run

x_train = np.linspace(-np.pi, np.pi, N_TRAIN)
x_eval  = np.linspace(-np.pi, np.pi, N_EVAL)
y_train = np.sin(x_train)
y_eval  = np.sin(x_eval)

QNLEP_METHOD_ORDER = ['data_reuploading', 'ancilla_tracing', 'ancilla_reuploading', 'strongly_entangling', 'qcnn', 'quantum_reservoir']

METHODS = {}
for name in QNLEP_METHOD_ORDER:
    if name == 'quantum_reservoir':
        METHODS[name] = QNLEP_BUILDERS[name](N_QUBITS, N_LAYERS, reservoir_seed=RUN_SEED)
    else:
        METHODS[name] = QNLEP_BUILDERS[name](N_QUBITS, N_LAYERS)
METHOD_BUILDERS = {name: QNLEP_BUILDERS[name] for name in QNLEP_METHOD_ORDER}
_all_noisy = bound_noisy_factories(N_QUBITS, N_LAYERS, reservoir_seed=RUN_SEED)
NOISY_FACTORIES = {name: _all_noisy[name] for name in QNLEP_METHOD_ORDER}


## 2. Noisy circuit factory

Implementations live in `src/circuits/noisy_qnlep.py`: for each QNLEP method we rebuild the same gate order on `default.mixed` and insert `qml.DepolarizingChannel` after gate groups. **Ancilla tracing** uses a **single** initial `RY(x)` on system qubits (matching the clean `ancilla_tracing` builder), not per-layer re-encoding.

In [ ]:
# Depolarizing-noise QNodes are defined in ``src/circuits/noisy_qnlep.py``
# (aligned with clean builders; ancilla uses a single initial encoding of x).
pass


## 3. Train all methods under zero noise (clean training)

**Caching:** Trained weights are saved to `results/nss_clean_train_cache.json` when `NSS_USE_CLEAN_TRAIN_CACHE` is True. The next time you run this cell, weights load from disk if `N_QUBITS`, `N_LAYERS`, `N_TRAIN`, `N_STEPS`, `LR`, the method list, and **`random_seed` (= `QNLEP_SEED` / `RUN_SEED`)** match the file (and `NSS_FORCE_CLEAN_RETRAIN` is False). Restarting the kernel clears RAM but not this file — re-executing §3 will then **load** instead of retraining. Delete the JSON or set `NSS_FORCE_CLEAN_RETRAIN = True` to train again.

In [ ]:
def _nss_clean_train_meta():
    return {
        "random_seed": RUN_SEED,
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
        "n_train": N_TRAIN,
        "n_eval": N_EVAL,
        "n_steps": N_STEPS,
        "lr": LR,
        "methods": list(METHODS.keys()),
    }


def _try_load_nss_clean_train():
    if not NSS_TRAIN_CACHE_PATH.is_file():
        return None
    with open(NSS_TRAIN_CACHE_PATH, encoding="utf-8") as f:
        data = json.load(f)
    if data.get("meta") != _nss_clean_train_meta():
        return None
    raw = data.get("trained_params", {})
    out = {}
    for name in METHODS:
        if name not in raw:
            return None
        w = np.asarray(raw[name], dtype=float)
        if w.shape != param_shape(name, N_QUBITS, N_LAYERS):
            return None
        out[name] = w
    return out


def _save_nss_clean_train(trained_params):
    payload = {
        "meta": _nss_clean_train_meta(),
        "trained_params": {k: np.asarray(v).tolist() for k, v in trained_params.items()},
    }
    with open(NSS_TRAIN_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


trained_params = {}
if NSS_USE_CLEAN_TRAIN_CACHE and not NSS_FORCE_CLEAN_RETRAIN:
    loaded = _try_load_nss_clean_train()
    if loaded is not None:
        trained_params = loaded
        print("Loaded clean-training weights from", NSS_TRAIN_CACHE_PATH)

if not trained_params:
    print("Training from scratch (no valid cache, or NSS_FORCE_CLEAN_RETRAIN=True)")
    for method_name, circuit in METHODS.items():
        shape = param_shape(method_name, N_QUBITS, N_LAYERS)
        weights = pnp.array(np.random.normal(0, 0.1, shape), requires_grad=True)

        def cost(w):
            preds = pnp.array([circuit(w, float(x)) for x in x_train])
            return pnp.mean((preds - y_train) ** 2)

        opt = qml.AdamOptimizer(stepsize=LR)
        for _ in range(N_STEPS):
            weights, _ = opt.step_and_cost(cost, weights)

        trained_params[method_name] = np.array(weights)

    if NSS_USE_CLEAN_TRAIN_CACHE:
        _save_nss_clean_train(trained_params)
        print("Saved clean-training weights to", NSS_TRAIN_CACHE_PATH)

for method_name, circuit in METHODS.items():
    w = trained_params[method_name]
    preds_clean = np.array([float(circuit(w, float(x))) for x in x_eval])
    print(f"{method_name:20s}  clean R²={r2_score(y_eval, preds_clean):.3f}")


## 4. Noise sweep: evaluate each method at each noise level

In [ ]:
r2_curves   = {}
skill_curves = {}
nss_scores  = {}
nss_detail  = {}

for method_name in METHODS:
    r2_curves[method_name] = []
    skill_curves[method_name] = []
    params  = trained_params[method_name]
    factory = NOISY_FACTORIES[method_name]

    for p_noise in NOISE_LEVELS:
        noisy_circuit = factory(p_noise)
        preds_noisy   = np.array([float(noisy_circuit(params, float(x))) for x in x_eval])
        r2            = r2_score(y_eval, preds_noisy)
        skill         = shifted_r2_score(r2)
        r2_curves[method_name].append(float(r2))
        skill_curves[method_name].append(float(skill))
        print(f"{method_name:20s}  p={p_noise:.3f}  R²={r2:.3f}  skill={skill:.3f}")

    nss = noise_stability_score(r2_curves[method_name], NOISE_LEVELS)
    nss_scores[method_name] = nss
    nss_detail[method_name] = {
        'r2_curve': [float(v) for v in r2_curves[method_name]],
        'skill_curve': [float(v) for v in skill_curves[method_name]],
        'clean_r2': float(r2_curves[method_name][0]),
        'clean_skill': float(skill_curves[method_name][0]),
        'score_definition': 'area_under_shifted_r2_skill_curve',
    }
    print(f"  → NSS = {nss:.3f}\n")


## 5. Visualizations

### 5a. R²(p) degradation curves per method

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = plt.cm.tab10.colors

for i, method_name in enumerate(METHODS):
    r2_vals = r2_curves[method_name]
    ax.plot(NOISE_LEVELS, r2_vals, 'o-', color=colors[i], lw=2,
            label=f"{method_name}  NSS={nss_scores[method_name]:.2f}")
    # shade area under the curve
    ax.fill_between(NOISE_LEVELS, r2_vals, alpha=0.08, color=colors[i])

ax.axhline(0.95, color='grey', ls=':', lw=1, label='R²=0.95 reference')
ax.set_xscale('symlog', linthresh=0.001)
ax.set_xlabel('Depolarizing noise rate p')
ax.set_ylabel('R²  (evaluated, frozen weights)')
ax.set_title('Noise stability: R²(p) degradation curves')
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/nss_r2_curves.pdf', bbox_inches='tight')
plt.show()


### 5b. NSS bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
method_names = list(METHODS.keys())
bars = ax.bar(method_names, [nss_scores[m] for m in method_names],
              color='steelblue', edgecolor='k')
ax.set_xticks(range(len(method_names)))
ax.set_xticklabels(method_names, rotation=25, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('NSS')
ax.set_title('Noise Stability Score')
for bar, m in zip(bars, method_names):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{nss_scores[m]:.2f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/nss_bar.pdf', bbox_inches='tight')
plt.show()


### 5c. Noisy predictions overlay at selected noise levels

Shows how the predicted function shape distorts as noise increases — gives
intuition beyond a single scalar score.

In [ ]:
selected_levels = [0.0, 0.01, 0.05, 0.1]
method_show     = list(METHODS.keys())[0]   # show first method; duplicate for others
factory_show    = NOISY_FACTORIES[method_show]
params_show     = trained_params[method_show]

fig, axes = plt.subplots(1, len(selected_levels), figsize=(3.5 * len(selected_levels), 3), sharey=True)

for ax, p_noise in zip(axes, selected_levels):
    c = factory_show(p_noise)
    p_arr = np.array([float(c(params_show, float(x))) for x in x_eval])
    ax.plot(x_eval, y_eval,   'k--', lw=1.5, label='target')
    ax.plot(x_eval, p_arr,    'C0',  lw=2.0, label=f'p={p_noise}')
    ax.set_title(f'p = {p_noise}', fontsize=9)
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel(method_show.replace('_', ' '))
fig.suptitle('Predicted function under increasing noise', y=1.02)
plt.tight_layout()
plt.savefig('../figures/nss_noisy_predictions.pdf', bbox_inches='tight')
plt.show()


## 6. Export NSS scores

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / 'nss_scores.json', 'w', encoding='utf-8') as f:
    json.dump(nss_scores, f, indent=2)
with open(RESULTS_DIR / 'r2_curves.json', 'w', encoding='utf-8') as f:
    json.dump(r2_curves, f, indent=2)
with open(RESULTS_DIR / 'nss_detail.json', 'w', encoding='utf-8') as f:
    json.dump({
        'score_definition': 'area_under_shifted_r2_skill_curve',
        'noise_levels': NOISE_LEVELS,
        'run_seed': RUN_SEED,
        'n_train': N_TRAIN,
        'n_eval': N_EVAL,
        'n_steps': N_STEPS,
        'by_method': nss_detail,
    }, f, indent=2)
print('Saved nss_scores.json, r2_curves.json, and nss_detail.json')
